# PhishGuard AI
## AI-Based Phishing Website Detection & Risk Analysis System
**B.Tech AI/ML College Project**

**Dataset:** PhiUSIIL Phishing URL Dataset - UCI Machine Learning Repository  
https://archive.ics.uci.edu/dataset/967/phiusiil+phishing+url+dataset

- Records: ~235,795 | Features: 54 | Labels: 0=Phishing, 1=Legitimate

## 1. Setup & Imports

In [ ]:
import sys, os, time, warnings
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Server-side rendering
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, precision_recall_curve, auc
)
import joblib
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
print('All imports successful!')

## 2. Load Dataset

In [ ]:
# Load the PhiUSIIL Phishing URL Dataset
# Download from: https://archive.ics.uci.edu/dataset/967/phiusiil+phishing+url+dataset
DATA_PATH = '../data/PhiUSIIL_Phishing_URL_Dataset.csv'
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
print(f'Columns ({len(df.columns)}): {df.columns.tolist()[:10]}...')
df.head()

In [ ]:
print('--- Data Types ---')
print(df.dtypes)
print('\n--- Basic Statistics ---')
df.describe()

## 3. Data Quality Check

In [ ]:
# Check missing values
print('--- Missing Values ---')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values found!')

# Check duplicates
n_dupes = df.duplicated().sum()
print(f'\nDuplicate rows: {n_dupes}')
if n_dupes > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f'Removed. New shape: {df.shape}')

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# 4.1 Class Distribution (Phishing vs Legitimate)
print('Class Distribution:')
counts = df['label'].value_counts()
for cls, count in counts.items():
    name = 'Legitimate' if cls == 1 else 'Phishing'
    pct = count/len(df)*100
    print(f'  {name} ({cls}): {count:,} ({pct:.2f}%)')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(['Legitimate (1)', 'Phishing (0)'],
            [counts.get(1,0), counts.get(0,0)],
            color=['#2ecc71', '#e74c3c'], edgecolor='black')
axes[0].set_title('Class Distribution (Count)', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate([counts.get(1,0), counts.get(0,0)]):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontweight='bold')
axes[1].pie([counts.get(1,0), counts.get(0,0)],
            labels=['Legitimate', 'Phishing'],
            colors=['#2ecc71', '#e74c3c'], autopct='%1.1f%%')
axes[1].set_title('Class Distribution (%)', fontweight='bold')
plt.suptitle('Phishing vs Legitimate URL Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../static/plots/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Interpretation: Near-balanced classes. Stratified splitting will maintain this ratio in train/test sets.')

In [ ]:
# 4.2 URL Length Distribution
if 'URLLength' in df.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    for label, color, name in [(0, '#e74c3c', 'Phishing'), (1, '#2ecc71', 'Legitimate')]:
        subset = df[df['label']==label]['URLLength']
        subset = subset[subset < subset.quantile(0.99)]  # Clip outliers
        ax.hist(subset, bins=50, alpha=0.6, color=color, label=name)
    ax.set_title('URL Length Distribution: Phishing vs Legitimate', fontweight='bold')
    ax.set_xlabel('URL Length (characters)')
    ax.set_ylabel('Frequency')
    ax.legend()
    plt.tight_layout()
    plt.savefig('../static/plots/url_length_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Interpretation: Phishing URLs tend to be longer to hide the real destination.')

In [ ]:
# 4.3 HTTPS Distribution
if 'IsHTTPS' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for idx, (label, color, name) in enumerate([(0, '#e74c3c', 'Phishing'), (1, '#2ecc71', 'Legitimate')]):
        subset = df[df['label']==label]['IsHTTPS'].value_counts(normalize=True)*100
        axes[idx].bar(['HTTP', 'HTTPS'], [subset.get(0,0), subset.get(1,0)],
                      color=['#95a5a6', color])
        axes[idx].set_title(f'{name} - HTTPS Usage', fontweight='bold')
        axes[idx].set_ylabel('Percentage (%)')
        for i, v in enumerate([subset.get(0,0), subset.get(1,0)]):
            axes[idx].text(i, v + 1, f'{v:.1f}%', ha='center', fontweight='bold')
    plt.suptitle('HTTPS Usage: Phishing vs Legitimate', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../static/plots/https_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Interpretation: HTTPS usage alone is not a safety guarantee.')

In [ ]:
# 4.4 Correlation Heatmap (Top Features)
numeric_df = df.select_dtypes(include=[np.number])
corr_with_label = numeric_df.corr()['label'].abs().sort_values(ascending=False)
top_feats = corr_with_label.index[:16].tolist()
corr_mat = numeric_df[top_feats].corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_mat, dtype=bool))
sns.heatmap(corr_mat, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, ax=ax, annot_kws={'size': 7}, linewidths=0.5)
ax.set_title('Correlation Heatmap - Top Features vs Label', fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig('../static/plots/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Interpretation: Features with high absolute correlation with label are most predictive.')

## 5. Feature Engineering

### Why These Features Matter for Phishing Detection

In [ ]:
feature_explanations = {
    'URLLength': 'Longer URLs hide real destination. Phishing sites average longer URLs.',
    'DomainLength': 'Very short/long domains indicate auto-generated or obfuscated names.',
    'NoOfSubDomain': 'Multiple subdomains (e.g., login.secure.bank.evil.com) are a phishing red flag.',
    'IsHTTPS': 'HTTPS is expected on legitimate sites. HTTP is a strong negative signal.',
    'NoOfOtherSpecialCharsInURL': 'Special chars confuse users and evade simple text filters.',
    'NoOfDash': 'Hyphens mimic legitimate brands (paypal-secure.com vs paypal.com).',
    'URLSimilarityIndex': 'Shannon entropy: high entropy = random/obfuscated URL characters.',
    'DomainTitleMatchScore': 'Low score = domain does not match page title = spoofing indicator.',
    'HasObfuscation': 'URL percent-encoding/hex values used to hide malicious content.',
    'Bank': 'Banking keywords used by phishers to appear legitimate.',
    'Pay': 'Payment keywords to lure victims.',
    'Crypto': 'Cryptocurrency keywords common in crypto scams.',
    'NoOfAt': '@ symbol in URL redirects browser to different host - classic trick.',
    'NoOfURLRedirects': 'Multiple redirects hide the final destination.',
}
print('Feature Explanations:')
print('-' * 80)
for feat, expl in feature_explanations.items():
    print(f'{feat:35s}: {expl}')

## 6. Data Preprocessing

In [ ]:
# Step 1: Remove irrelevant raw-text columns (not useful for ML)
drop_cols = [col for col in ['FILENAME', 'Domain', 'URL', 'Title'] if col in df.columns]
if drop_cols:
    df_clean = df.drop(columns=drop_cols)
    print(f'Dropped raw text columns: {drop_cols}')
else:
    df_clean = df.copy()

# Step 2: Encode categorical columns
cat_cols = [c for c in df_clean.select_dtypes(include=['object']).columns if c != 'label']
if cat_cols:
    le = LabelEncoder()
    for col in cat_cols:
        df_clean[col] = le.fit_transform(df_clean[col].astype(str))
    print(f'Encoded categorical columns: {cat_cols}')
else:
    print('No categorical columns to encode (all numeric).')

# Step 3: Check class distribution
print(f'\nClass distribution: {df_clean["label"].value_counts().to_dict()}')

# Step 4: Stratified Train-Test Split (80/20)
X = df_clean.drop(columns=['label'])
y = df_clean['label']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y  # Stratified!
)
print(f'\nTrain set: {X_train.shape[0]} samples')
print(f'Test set:  {X_test.shape[0]} samples')
print(f'Train class balance: {y_train.value_counts().to_dict()}')
print(f'Test class balance:  {y_test.value_counts().to_dict()}')

# Step 5: Feature Scaling
# IMPORTANT: Fit scaler ONLY on training data to prevent data leakage!
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)  # Fit + transform on train
X_test_sc = scaler.transform(X_test)          # Transform only on test
print('\nStandardScaler fitted on training data only.')
print('Test set scaled using training statistics (no data leakage).')

## 7. Model Training & Evaluation

In [ ]:
# Define all models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1),
    'Decision Tree': DecisionTreeClassifier(max_depth=15, min_samples_split=20, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42),
}

try:
    from xgboost import XGBClassifier
    models['XGBoost'] = XGBClassifier(n_estimators=100, learning_rate=0.1,
        max_depth=6, eval_metric='logloss', random_state=42, n_jobs=-1)
    print('XGBoost available and added.')
except ImportError:
    print('XGBoost not available, skipping.')

# Train and evaluate all models
results = {}
print('\n' + '='*70)
print('  MODEL TRAINING AND EVALUATION')
print('='*70)

for name, model in models.items():
    print(f'\nTraining: {name}...')
    t0 = time.time()
    model.fit(X_train_sc, y_train)
    train_time = time.time() - t0

    t1 = time.time()
    y_pred = model.predict(X_test_sc)
    pred_time = time.time() - t1
    y_prob = model.predict_proba(X_test_sc)[:,1] if hasattr(model, 'predict_proba') else y_pred

    results[name] = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, y_prob),
        'y_pred': y_pred, 'y_prob': y_prob,
        'model': model, 'train_time': train_time, 'pred_time': pred_time
    }
    m = results[name]
    print(f'  Accuracy  : {m["accuracy"]:.4f}')
    print(f'  Precision : {m["precision"]:.4f}')
    print(f'  Recall    : {m["recall"]:.4f}')
    print(f'  F1-Score  : {m["f1"]:.4f}')
    print(f'  ROC-AUC   : {m["roc_auc"]:.4f}')
    print(f'  Train time: {train_time:.2f}s | Pred time: {pred_time:.4f}s')

In [ ]:
# Summary Table
print(f'\n{"Model":25s} | Accuracy | Precision | Recall | F1-Score | ROC-AUC | Train(s)')
print('-'*90)
for name, m in results.items():
    print(f'{name:25s} | {m["accuracy"]:.4f}   | {m["precision"]:.4f}    | '
          f'{m["recall"]:.4f} | {m["f1"]:.4f}   | {m["roc_auc"]:.4f}  | {m["train_time"]:.1f}s')

In [ ]:
# ROC Curves - all models
fig, ax = plt.subplots(figsize=(9, 7))
colors = plt.cm.tab10(np.linspace(0, 1, len(results)))
for (name, m), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, m['y_prob'])
    ax.plot(fpr, tpr, color=color, lw=2, label=f"{name} (AUC={m['roc_auc']:.4f})")
ax.plot([0,1],[0,1],'k--',lw=1.5,label='Random Classifier')
ax.fill_between([0,1],[0,1], alpha=0.05, color='gray')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate (Recall)', fontsize=12)
ax.set_title('ROC Curves - All Models Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('../static/plots/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Interpretation: Curve closest to top-left corner has best discriminative power.')

In [ ]:
# Precision-Recall Curves
fig, ax = plt.subplots(figsize=(9, 7))
colors = plt.cm.tab10(np.linspace(0, 1, len(results)))
for (name, m), color in zip(results.items(), colors):
    prec, rec, _ = precision_recall_curve(y_test, m['y_prob'])
    pr_auc = auc(rec, prec)
    ax.plot(rec, prec, color=color, lw=2, label=f"{name} (PR-AUC={pr_auc:.4f})")
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curves - All Models', fontsize=14, fontweight='bold')
ax.legend(loc='lower left', fontsize=9)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('../static/plots/pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion Matrices for all models
n = len(results)
cols_per_row = 3
rows = (n + cols_per_row - 1) // cols_per_row
fig, axes = plt.subplots(rows, cols_per_row, figsize=(15, 5*rows))
if rows == 1:
    axes = [axes] if n == 1 else list(axes)
else:
    axes = [ax for row in axes for ax in row]

for ax, (name, m) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, m['y_pred'])
    disp = ConfusionMatrixDisplay(cm, display_labels=['Phishing','Legitimate'])
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(name, fontweight='bold')

for ax in axes[len(results):]:
    ax.set_visible(False)

plt.suptitle('Confusion Matrices - All Models', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../static/plots/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('FN (bottom-left) = phishing missed = most dangerous error in phishing detection.')

In [ ]:
# Model Comparison Bar Chart
metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
metric_labels = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
model_names = list(results.keys())
x = np.arange(len(metrics_to_plot))
width = 0.8 / len(model_names)
colors = plt.cm.tab10(np.linspace(0, 1, len(model_names)))

fig, ax = plt.subplots(figsize=(13, 7))
for i, (name, color) in enumerate(zip(model_names, colors)):
    values = [results[name][m] for m in metrics_to_plot]
    ax.bar(x + i*width - (len(model_names)-1)*width/2, values,
           width, label=name, color=color, alpha=0.85, edgecolor='black', lw=0.5)

ax.set_xlabel('Metric', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance Comparison - All Metrics', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metric_labels, fontsize=11)
ax.set_ylim(0, 1.08)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.4)
ax.axhline(y=0.9, color='red', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('../static/plots/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature Importance - Random Forest (or best tree-based model)
fi_model_name = next((n for n in ['Random Forest', 'Gradient Boosting', 'XGBoost', 'Decision Tree']
                      if n in results), None)
if fi_model_name:
    fi_model = results[fi_model_name]['model']
    importances = fi_model.feature_importances_
    feat_names = X_train.columns.tolist()
    indices = np.argsort(importances)[-20:]

    fig, ax = plt.subplots(figsize=(10, 8))
    colors_fi = plt.cm.YlOrRd(importances[indices]/importances[indices].max())
    bars = ax.barh(range(len(indices)), importances[indices],
                   color=colors_fi, edgecolor='black', lw=0.5)
    ax.set_yticks(range(len(indices)))
    ax.set_yticklabels([feat_names[i] for i in indices], fontsize=8)
    ax.set_xlabel('Feature Importance Score', fontsize=12)
    ax.set_title(f'Top 20 Feature Importances - {fi_model_name}', fontsize=13, fontweight='bold')
    ax.grid(axis='x', alpha=0.4)
    for bar, val in zip(bars, importances[indices]):
        ax.text(val + 0.001, bar.get_y() + bar.get_height()/2, f'{val:.4f}', va='center', fontsize=7)
    plt.tight_layout()
    plt.savefig(f'../static/plots/feature_importance_{fi_model_name.replace(" ","_").lower()}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

## 8. Model Selection - Weighted Composite Score

In [ ]:
# Multi-criteria model selection
# Composite Score = 0.40*F1 + 0.30*ROC_AUC + 0.20*Recall + 0.10*Precision
#
# Rationale:
# - F1 (40%): Primary metric - balances precision and recall
# - ROC-AUC (30%): Overall discriminative power
# - Recall (20%): Critical for phishing - missing a phishing site is dangerous!
# - Precision (10%): Minimize false alarms

print('Model Selection Methodology')
print('Score = 0.40*F1 + 0.30*ROC_AUC + 0.20*Recall + 0.10*Precision')
print('\nRationale: In phishing detection, a False Negative (missed phishing) is')
print('far more dangerous than a False Positive (legitimate site blocked).')
print('=' * 70)

scores = {}
for name, m in results.items():
    score = 0.40*m['f1'] + 0.30*m['roc_auc'] + 0.20*m['recall'] + 0.10*m['precision']
    scores[name] = score
    print(f'{name:25s} | Composite: {score:.4f} | F1={m["f1"]:.4f} | '
          f'AUC={m["roc_auc"]:.4f} | Recall={m["recall"]:.4f}')

best = max(scores, key=scores.get)
print(f'\n>>> SELECTED MODEL: {best} (Score={scores[best]:.4f}) <<<')

In [ ]:
# Save best model and all artifacts
import json
os.makedirs('../models', exist_ok=True)

joblib.dump(results[best]['model'], '../models/final_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(X_train.columns.tolist(), '../models/feature_names.pkl')

# Save metrics as JSON for web app
metrics_out = {}
for name, m in results.items():
    metrics_out[name] = {
        'accuracy': float(m['accuracy']),
        'precision': float(m['precision']),
        'recall': float(m['recall']),
        'f1': float(m['f1']),
        'roc_auc': float(m['roc_auc']),
        'train_time_s': round(float(m['train_time']), 2),
        'prediction_time_s': round(float(m['pred_time']), 4),
        'confusion_matrix': confusion_matrix(y_test, m['y_pred']).tolist()
    }
metrics_out['_best_model'] = best

with open('../models/metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)

print(f'Saved artifacts:')
print(f'  - models/final_model.pkl  ({best})')
print(f'  - models/scaler.pkl')
print(f'  - models/feature_names.pkl')
print(f'  - models/metrics.json')

## 9. Case Study - Detection of a Suspicious Login Website

In [ ]:
# CASE STUDY: Detection of a Suspicious Login Website
# DISCLAIMER: This is a simulated educational example.
# The URL below is fictional and not claimed to be an actual malicious website.

print('=' * 60)
print('CASE STUDY: Detection of a Suspicious Login Website')
print('=' * 60)
print()
print('SCENARIO:')
print('A user receives an email claiming their account needs urgent')
print('verification and is provided the following link:')
print()

sys.path.insert(0, '../src')
from predict import predict_url

case_url = 'https://secure-account-verification-example.com/login'
print(f'URL: {case_url}')
print()

result = predict_url(case_url)
if result.get('error'):
    print(f'Error: {result["error"]}')
    print('Note: Train the model first using run_training.py')
else:
    print('--- ML SYSTEM ANALYSIS RESULT ---')
    print(f'Prediction  : {result["prediction"]}')
    print(f'Risk Score  : {result["risk_score"]}%')
    print(f'Confidence  : {result["confidence"]}%')
    print(f'Risk Level  : {result["risk_level"]}')
    print()
    print('Top Contributing Factors:')
    for i, f in enumerate(result['contributing_factors'], 1):
        print(f'  {i}. {f}')
    print()
    print('Feature Analysis:')
    feats = result['interpretable_features']
    for k, v in feats.items():
        if not isinstance(v, list) or v:
            print(f'  {k:30s}: {v}')
    print()
    print(f'Recommendation: {result["safety_recommendation"]}')
    print()
    print('DISCLAIMER: Simulated educational example only.')
    print('The domain above is fictional. This system analyzes URL structure only.')

## 10. Conclusion

This notebook demonstrates a complete end-to-end ML pipeline for phishing URL detection:

- ✅ Loaded and explored 235,795 URLs from the PhiUSIIL dataset
- ✅ Performed comprehensive EDA with 6+ visualization types
- ✅ Feature engineering: 40+ features with phishing-relevance explanations
- ✅ Preprocessing: Stratified split, StandardScaler (no data leakage)
- ✅ Trained 4-5 ML models (LR, DT, RF, GB, XGB)
- ✅ Full evaluation: Accuracy, F1, Recall, Precision, ROC-AUC, Confusion Matrix
- ✅ Documented model selection with weighted composite score methodology
- ✅ Saved final model + preprocessing artifacts
- ✅ Case study with explainable AI output

**To start the web application:**
```bash
python app.py
# Access at http://127.0.0.1:5000
```

---
*PhishGuard AI | B.Tech AI/ML Project 2024*